In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 72.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import chromadb
import uuid

# ==========================================
# 0. HARDWARE SETUP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** System running on: {device.type.upper()} ***")

# ==========================================
# 1. THE ENVIRONMENT
# ==========================================
class SimpleGridEnv:
    def __init__(self, size=15, num_food=40, num_hazards=10):
        self.size = size
        self.num_food = num_food
        self.num_hazards = num_hazards
        self.reset()

    def reset(self):
        self.agent_pos = [self.size // 2, self.size // 2]
        self.grid = np.zeros((self.size, self.size))
        
        for _ in range(self.num_food):
            self.grid[np.random.randint(0, self.size), np.random.randint(0, self.size)] = 1
        for _ in range(self.num_hazards):
            self.grid[np.random.randint(0, self.size), np.random.randint(0, self.size)] = -1
            
        self.grid[self.agent_pos[0], self.agent_pos[1]] = 0 
        return self.get_fov()

    def get_fov(self):
        r, c = self.agent_pos
        fov = np.full((5, 5), -2.0) 
        for i in range(-2, 3):
            for j in range(-2, 3):
                if 0 <= r+i < self.size and 0 <= c+j < self.size:
                    fov[i+2, j+2] = self.grid[r+i, c+j]
        return torch.FloatTensor(fov.flatten())

    def step(self, action_idx):
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
        dr, dc = moves[action_idx]
        
        new_r = np.clip(self.agent_pos[0] + dr, 0, self.size - 1)
        new_c = np.clip(self.agent_pos[1] + dc, 0, self.size - 1)
        self.agent_pos = [new_r, new_c]
        
        reward = self.grid[new_r, new_c]
        if reward == 1: 
            self.grid[new_r, new_c] = 0 
        
        return self.get_fov(), reward

# ==========================================
# 2. THE ARCHITECTURE (Split-Brain)
# ==========================================
class EvaluativeBrain(nn.Module):
    def __init__(self):
        super().__init__()
        # Instinct (Evolves)
        self.actor = nn.Sequential(nn.Linear(25, 16), nn.ReLU(), nn.Linear(16, 4))
        # Wisdom (Learns)
        self.critic = nn.Sequential(nn.Linear(25, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, state_tensor):
        return self.actor(state_tensor), self.critic(state_tensor)

# ==========================================
# 3. CULTURE DATABASE INITIALIZATION
# ==========================================
chroma_client = chromadb.Client()
try: chroma_client.delete_collection("tribe_culture")
except: pass

culture_db = chroma_client.create_collection(
    name="tribe_culture", 
    metadata={"hnsw:space": "l2"} 
)
# Seed vector to prevent initial crash
culture_db.add(embeddings=[[0.0]*25], metadatas=[{"action": 0, "reward": 0.0, "usage_count": 0}], ids=["genesis_memory"])

# ==========================================
# 4. THE GRAND TRAINING LOOP
# ==========================================
NUM_AGENTS = 50
GENERATIONS = 100
LIFESPAN = 150
LEARNING_RATE = 0.005
GAMMA = 0.95

population = [EvaluativeBrain().to(device) for _ in range(NUM_AGENTS)]
history_fitness, history_queries, history_writes = [], [], []

print("Starting Tribal RAG Evolution...")

for gen in range(GENERATIONS):
    fitness_scores = []
    generation_queries, successful_writes = 0, 0
    batch_embeddings, batch_metadatas, batch_ids = [], [], []
    
    for agent in population:
        env = SimpleGridEnv()
        state = env.reset().to(device)
        optimizer = optim.Adam(agent.parameters(), lr=LEARNING_RATE)
        values, rewards, agent_memory_buffer = [], [], []
        
        # --- LIFESPAN (ACT & QUERY) ---
        for step in range(LIFESPAN):
            action_logits, value = agent(state)
            final_action = torch.argmax(action_logits).item()
            
            # The Query Gate (Instinct Veto)
            if value.item() < 0.2: 
                generation_queries += 1
                results = culture_db.query(
                    query_embeddings=[state.cpu().numpy().tolist()], n_results=1,
                    where={"reward": {"$gt": 0.0}}
                )
                if results['metadatas'][0]:
                    final_action = results['metadatas'][0][0]['action']
                    
            next_state, reward = env.step(final_action)
            
            if reward > 0:
                agent_memory_buffer.append({"state": state.cpu().numpy().tolist(), "action": final_action, "reward": reward})
            
            state = next_state.to(device)
            values.append(value)
            rewards.append(reward)
            
        # --- LEARNING (BACKPROP) ---
        returns = []
        R = 0
        for r in reversed(rewards):
            R = r + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        values = torch.cat(values).squeeze()
        
        loss = nn.MSELoss()(values, returns)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step() 
        
        total_lifespan_reward = sum(rewards)
        fitness_scores.append(total_lifespan_reward)
        
        # --- WRITING TO CULTURE ---
        if total_lifespan_reward > 3.0: # The lowered threshold
            for mem in agent_memory_buffer:
                batch_embeddings.append(mem["state"])
                batch_metadatas.append({"action": mem["action"], "reward": mem["reward"], "usage_count": 0})
                batch_ids.append(str(uuid.uuid4()))
                successful_writes += 1
                
    if batch_embeddings:
        culture_db.add(embeddings=batch_embeddings, metadatas=batch_metadatas, ids=batch_ids)
    
    # --- EVOLUTION (LAMARCKIAN + DARWINIAN) ---
    avg_fitness = np.mean(fitness_scores)
    history_fitness.append(avg_fitness)
    history_queries.append(generation_queries)
    history_writes.append(successful_writes)
    
    sorted_indices = np.argsort(fitness_scores)[::-1]
    survivors = [deepcopy(population[i]) for i in sorted_indices[:NUM_AGENTS // 2]]
    next_generation = deepcopy(survivors)
    
    while len(next_generation) < NUM_AGENTS:
        parent1 = survivors[np.random.randint(0, len(survivors))]
        parent2 = survivors[np.random.randint(0, len(survivors))]
        child = EvaluativeBrain().to(device)
        with torch.no_grad():
            for (name_p1, p1), (name_p2, p2), (name_c, c) in zip(parent1.named_parameters(), parent2.named_parameters(), child.named_parameters()):
                if 'actor' in name_c: # Mutate Instinct
                    mask = torch.rand_like(p1) > 0.5
                    c.copy_(torch.where(mask, p1, p2))
                    c.add_((torch.rand_like(c) < 0.1) * torch.randn_like(c) * 0.1)
                elif 'critic' in name_c: # Inherit Wisdom
                    c.copy_(p1)
        next_generation.append(child)
    population = next_generation
    
    if (gen + 1) % 10 == 0:
        print(f"Gen {gen+1:3d} | Avg Fitness: {avg_fitness:5.2f} | DB Size: {culture_db.count():5d} | Queries: {generation_queries:4d} | Writes: {successful_writes:4d}")

# ==========================================
# 5. VISUALIZATION
# ==========================================
fig, ax1 = plt.subplots()
ax1.set_xlabel('Generation')
ax1.set_ylabel('Average Fitness', color='tab:blue')
ax1.plot(history_fitness, color='tab:blue', label="Fitness")
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()  
ax2.set_ylabel('Culture Queries', color='tab:red')  
ax2.plot(history_queries, color='tab:red', linestyle='dashed', label="Queries")
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Tribal RAG: Fitness vs. Cultural Reliance')
fig.tight_layout()  
plt.show()

*** System running on: CUDA ***
Starting Tribal RAG Evolution...
Gen  10 | Avg Fitness:  2.82 | DB Size:  1578 | Queries: 4863 | Writes:  256
Gen  20 | Avg Fitness:  0.90 | DB Size:  3674 | Queries: 5090 | Writes:  152


KeyboardInterrupt: 